# Delta Lake & Unity Catalog Foundations

## Reference domain

We continue with the **Fintech consumer bank** from `../Fintech`. The two tables driving this notebook are `silver.customers` (the shared dimension you stood up at the end of notebook 01) and a new `silver.card_transactions` — the busiest fact table in the bank, the one fraud and ML teams care about most. Everything we do — ACID, time travel, MERGE, OPTIMIZE — is shown against these tables.

## The problem — what raw Parquet on object storage does not give you

Picture the bank's pipeline before Delta Lake existed. The card-transactions feed lands as Parquet files on S3 every fifteen minutes. The fraud dashboard reads the same path. Three things keep breaking.

**Partial writes are visible.** A Spark job writing the 10:15 batch creates twenty Parquet files in a partition. If the cluster dies after fifteen are written, the dashboard sees fifteen new files alongside the old ones — no transaction wrapped them, so the partial result is exposed.

**Concurrent writers corrupt each other.** Two jobs land late-arriving data into the same hour. They both list the partition, both write new files, both rewrite the partition's `_SUCCESS` marker. There is no lock. The final state depends on who finishes last.

**No schema enforcement.** The Cards source team adds a `merchant_country` column. The next batch's Parquet files have eleven columns; the existing files have ten. Spark inferring the schema picks the first file it sees, and either silently drops the new column for everyone or fails the read with a cryptic mismatch.

Plus, on top of those, no time travel (you cannot ask "what did this table look like yesterday at 9 a.m. before the bad backfill?"), no efficient row-level updates, and no audit log of who changed what. You can build all of that yourself — many teams did, badly. Delta Lake is the open-source library that gives you all of it as one transaction log.

## What Delta Lake actually is

A **Delta table** on disk is two things in one directory:

```text
 s3://fintech/silver/card_transactions/
 ├── _delta_log/                    ← the transaction log
 │   ├── 00000000000000000000.json
 │   ├── 00000000000000000001.json
 │   ├── 00000000000000000002.json
 │   └── 00000000000000000010.checkpoint.parquet
 ├── part-00000-...-c000.snappy.parquet     ← the data
 ├── part-00001-...-c000.snappy.parquet
 └── part-00002-...-c000.snappy.parquet
```

**The Parquet files** are normal column-oriented data files — anything that reads Parquet can read them.

**The `_delta_log` directory** holds an ordered sequence of JSON commit files, one per transaction. Each commit lists actions: "add this file", "remove that file", "the schema is now this", "the table properties are now those". A reader replays the log to compose the current table state — *which files belong, which don't, and what columns they have*.

Every Delta operation is the same loop: **plan the change → write new Parquet files → atomically append a new JSON commit to `_delta_log`**. The atomic append is the trick that turns a pile of files on object storage into a real transactional table.

## The transaction log — what a commit file actually contains

Each `NNNN.json` file is a small JSON document with a few action types. Skipping boilerplate, here's the shape of what a single commit might say:

```text
 00000000000000000003.json   (the third commit on this table)
  ┌──────────────────────────────────────────────────────────┐
  │  { "commitInfo": { "operation": "WRITE",                  │
  │                    "userName": "...",                     │
  │                    "timestamp": 17... } }                 │
  │  { "add":    { "path": "part-00000-...parquet",          │
  │                "size": 12834, "stats": "{...}" } }       │
  │  { "add":    { "path": "part-00001-...parquet", ... } }  │
  │  { "remove": { "path": "part-00007-...parquet",          │
  │                "deletionTimestamp": ... } }              │
  └──────────────────────────────────────────────────────────┘
```

The three actions that matter most:

- **`add`** — a Parquet file is now part of the table. The `stats` field embeds min/max/null-count per column, which is how Delta does file-skipping at read time without opening every file.
- **`remove`** — a Parquet file is no longer part of the table. The file is *not deleted from object storage* — it just stops being listed. `VACUUM` is what actually removes the bytes later.
- **`metaData`** — the table schema, partitioning, and properties. Schema changes emit a new `metaData` action.

Every ten commits (default), Delta writes a **checkpoint** — a Parquet file that snapshots the entire log up to that point. Readers load the latest checkpoint plus any JSON commits after it, instead of replaying all history. This is what keeps reads fast as the log grows.

## ACID on object storage — the four properties, paid for by the log

Object stores like S3 don't give you transactions natively. Delta Lake builds them up:

- **Atomicity** — a transaction commits when the next `NNNN.json` file appears in `_delta_log`. The JSON write itself is atomic (a single PUT). Either readers see the new commit file, or they don't. Half-written data files written *before* the commit are not part of the table because no `add` action references them yet.
- **Consistency** — every commit is validated against the current table state (schema, constraints) before it's accepted.
- **Isolation** — Delta uses **optimistic concurrency control**. Two writers prepare their commits independently; whoever wins the race to write `NNNN.json` first wins. The loser re-reads the latest log, checks whether its changes still apply (no conflict on touched files), and retries.
- **Durability** — the underlying object store handles durability for the data files and the log files. Once `NNNN.json` is written, it's persisted.

**Why this is exam-relevant:** the bank's nightly card pipeline and the real-time fraud stream can write to the same table without corrupting each other. Either commit succeeds and is visible to readers, or it conflicts and retries — never a partial result.

## Time travel — reading an old version of a table

Because the log is *append-only*, every past state of the table still exists — as long as the data files it referenced haven't been `VACUUM`-ed. You query a past state by version number or timestamp.

Three operations:

- **`VERSION AS OF n`** — read the table as it was at commit `n`. Useful for reproducibility ("what data did the model train on?").
- **`TIMESTAMP AS OF '2026-06-18 09:00:00'`** — read the table as it was at that wall-clock time. Useful for incident response ("what did the dashboard show before the bad backfill?").
- **`RESTORE TABLE ... TO VERSION AS OF n`** — write a new commit that makes the *current* state equal to a past version. Time travel rewinds your read; `RESTORE` rewinds the table itself.

**Retention** is governed by two table properties:

- `delta.logRetentionDuration` (default 30 days) — how long old commit JSONs are kept.
- `delta.deletedFileRetentionDuration` (default 7 days) — how long files marked `remove` are kept before `VACUUM` may delete them.

If you `VACUUM` aggressively, you lose the ability to time-travel further back than the retention setting. Be deliberate.

## Schema enforcement vs. schema evolution

Delta Lake enforces the table's schema on every write. By default, a write that introduces a new column fails — that's **schema enforcement**, and it's what keeps the Cards table from silently growing a `merchant_country` column from a misbehaving upstream.

When the column is intentional, you opt in to **schema evolution**:

- **`mergeSchema = true`** on a write — adds missing columns from the incoming DataFrame to the table. Existing rows get `NULL` in the new columns.
- **`overwriteSchema = true`** on an `INSERT OVERWRITE` — replaces the schema entirely. Destructive; use rarely.
- **`ALTER TABLE ... ADD COLUMNS (...)`** — explicit DDL. The most auditable option for production.

Auto Loader (notebook 03) has its own schema-evolution modes — `addNewColumns`, `rescue`, `failOnNewColumns`, `none` — built on the same Delta primitive.

**The rule that ships to the exam:** schema enforcement is **on by default**. Evolution is **opt-in per write** or via explicit `ALTER TABLE`. Silent column additions are not a thing on Delta.

## `MERGE INTO` — upserts, deletes, and SCD patterns

`MERGE INTO` is the single most-used Delta DML statement in production. It joins a *source* (incoming changes) to a *target* (the table) on a key, and lets you express what to do in each case:

```sql
 MERGE INTO fintech_dev.silver.customers AS t
 USING staging_customers_today          AS s
    ON t.customer_id = s.customer_id
 WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
 WHEN NOT MATCHED THEN INSERT *
 WHEN NOT MATCHED BY SOURCE THEN DELETE       -- optional: hard-delete missing customers
```

Two patterns the bank uses constantly.

**SCD Type 1 (overwrite)** — the customer's `city` field changes; just update it in place. Use `WHEN MATCHED ... UPDATE SET *`. No history kept.

**SCD Type 2 (history)** — the customer's `city` changes; close the old row with `valid_to = today` and `is_current = false`, then insert a new row with `valid_from = today` and `is_current = true`. Implemented as two `MERGE` passes: one to close existing rows (`WHEN MATCHED AND values_changed THEN UPDATE SET is_current = false`), and a separate `INSERT` of the new active row. Notebook 05 walks through the full pattern in a pipeline.

`MERGE` is also how Change Data Capture (CDC) feeds land into bronze and silver. `APPLY CHANGES INTO` in Lakeflow Spark Declarative Pipelines (notebook 05) is essentially a declarative wrapper around the same `MERGE` machinery.

## `UPDATE`, `DELETE`, and deletion vectors

Delta supports row-level `UPDATE` and `DELETE` against tables on immutable object storage. Two strategies under the hood:

**Copy-on-write (classic)** — Delta finds every Parquet file containing matching rows, *rewrites* each file without those rows (or with the updated values), and emits `add` + `remove` actions in the commit. Correct but expensive when only a few rows in a 500 MB file need changing.

**Deletion vectors (modern, default on supported runtimes)** — instead of rewriting the file, Delta writes a small bitmap that says *"skip rows 17 and 234 in this file."* The original Parquet file stays put; the deletion vector rides alongside it in the log. Reads apply the bitmap on the fly. The next `OPTIMIZE` compacts the file and resolves the deletions for real.

**Why this matters for the bank:** GDPR "right to be forgotten" and DSAR-style row deletes used to require rewriting gigabytes per request. With deletion vectors enabled, the delete is a small commit, and the actual file rewrites happen lazily during maintenance windows.

## Table maintenance — `OPTIMIZE`, `ZORDER`, `VACUUM`

Three commands every Delta table needs over its life.

**`OPTIMIZE`** — compacts many small files into fewer ~1 GB target-size files. Streaming writes and CDC merges create lots of small files; small files mean slow reads. `OPTIMIZE` is a metadata-level rewrite that reduces file count without changing what the table contains.

**`ZORDER BY (col_a, col_b)`** — co-locates rows with similar values of the listed columns into the same files. This dramatically improves file-skipping at read time when queries filter on those columns. Example: `ZORDER BY (customer_id)` on `card_transactions` lets per-customer lookups read 1% of files instead of all of them.

**`VACUUM`** — actually deletes the data files marked `remove` more than `deletedFileRetentionDuration` ago (default 7 days). This is what reclaims storage cost. **Important:** the default retention is enforced — Delta blocks shorter retentions unless you explicitly opt in, because shorter retention can break in-flight readers and time travel.

```sql
 OPTIMIZE fintech_dev.silver.card_transactions
   ZORDER BY (customer_id, transaction_at);

 VACUUM fintech_dev.silver.card_transactions RETAIN 168 HOURS;   -- 7 days, the default
```

On Unity Catalog, **Predictive Optimization** can run these for you automatically — covered next.

## Liquid Clustering — the modern replacement for partitioning + `ZORDER`

Classic Delta layouts force a one-time decision: partition by `transaction_date`? Partition by `customer_id`? Get it wrong and you're stuck — re-partitioning a TB-scale table is a full rewrite.

**Liquid Clustering** replaces both partitioning *and* `ZORDER` with a single property: declare which columns the table is clustered on, and Delta organises data into clusters that can be incrementally re-balanced. You can change the cluster keys later without rewriting the world.

```sql
 CREATE TABLE fintech_dev.silver.card_transactions (...)
   CLUSTER BY (customer_id, transaction_at);

 -- Change keys later — Delta will reorganise incrementally
 ALTER TABLE fintech_dev.silver.card_transactions
   CLUSTER BY (merchant_category, transaction_at);
```

**What you give up:** the explicit directory-style partition structure. **What you gain:** no small-files-per-partition problem, no skew when one partition is 100× larger than another, and the ability to evolve clustering as query patterns change.

**For the exam — remember the trade and the recommendation:** Liquid Clustering is recommended for new Delta tables. Mentioned again in notebook 08.

## Predictive Optimization

On Unity Catalog managed tables, **Predictive Optimization** runs `OPTIMIZE` and `VACUUM` for you, on a schedule informed by your actual access patterns. You enable it at the catalog or schema level, and Databricks does the right amount of compaction and cleanup, billed at a serverless rate.

```sql
 ALTER CATALOG fintech_dev
   ENABLE PREDICTIVE OPTIMIZATION;
```

Two reasons this matters for an associate-level exam:

- It's the **"correct answer"** when a question asks how to keep curated tables performant without writing a maintenance job.
- It's **Unity Catalog only** — external tables stored in raw paths outside UC do not get this. Yet another reason to prefer managed tables.

## Delta UniForm — interoperate with Iceberg without copying

Some downstream tools speak Iceberg, not Delta. **Delta UniForm** writes an Iceberg metadata layer *alongside* the Delta `_delta_log`, pointing at the same Parquet files. One physical dataset, two readable formats.

```sql
 ALTER TABLE fintech_dev.silver.card_transactions
   SET TBLPROPERTIES ('delta.universalFormat.enabledFormats' = 'iceberg');
```

Use it when: Snowflake, BigQuery, Trino, or any Iceberg-native tool needs to read tables you produce in Databricks. You don't double-write or maintain a separate Iceberg pipeline.

# Unity Catalog

Delta Lake makes the *files on disk* trustworthy. **Unity Catalog (UC)** makes the *tables, views, volumes, models, and functions* discoverable and governed. One catalog across the whole platform — for SQL, Python, Scala, ML, dashboards, and external query engines via federation.

Three things UC gives you that the legacy hive metastore could not:

- **A unified permission model** — GRANT / REVOKE on principals (users, groups, service principals) at any level of the hierarchy.
- **Lineage and audit out of the box** — every read and write is tracked, every job's column-level lineage is visible.
- **Cross-workspace, cross-cloud namespacing** — one metastore per region, shared by every workspace in that region.

## The Unity Catalog object hierarchy

```text
 metastore (one per region)
  │
  ├── catalog: fintech_dev
  │    ├── schema: bronze
  │    │    ├── table: card_transactions_raw
  │    │    ├── view:  card_transactions_recent
  │    │    ├── volume: landing_zone           ← files, not rows
  │    │    └── function: parse_merchant
  │    ├── schema: silver
  │    └── schema: gold
  └── catalog: ml_models
       └── schema: fraud
            └── model: card_fraud_v3           ← MLflow model in UC
```

Six **securable** object types live at the schema level: **tables, views, volumes, functions, models, and materialized views**. Each is referenced by its three-part name.

Permissions flow down the hierarchy. Granting `USE CATALOG` on `fintech_dev` plus `SELECT` on `fintech_dev.gold` lets a principal read every table in gold without touching bronze or silver. We cover GRANT semantics fully in notebook 09 — this notebook focuses on the *shape* of the namespace.

## Managed vs. external tables — the choice that decides everything else

Every UC table is one of two flavours.

**Managed table.** Unity Catalog owns the metadata *and* the underlying files. Files live at a UC-managed storage location associated with the catalog or schema. Drop the table, and the files are eventually deleted.

**External table.** UC owns the metadata; the files live at a path *you* specify, inside a UC **external location**. Drop the table, and only the metadata is removed — the files stay.

| Question | Managed | External |
|---|---|---|
| Who owns the files? | Unity Catalog | You |
| Where do files live? | UC-managed location | A path you specify |
| What does DROP TABLE do? | Drops metadata; files deleted after retention | Drops metadata only; files preserved |
| Gets Predictive Optimization? | ✅ | ❌ |
| Gets full UC lifecycle features? | ✅ | Partial |
| Used when files must be shared with non-Databricks tools? | ❌ | ✅ |
| Used when you want UC to manage everything end to end? | ✅ | ❌ |

**The default the exam rewards: prefer managed.** External tables exist for specific reasons — you have a multi-tool data lake, the files are produced by something other than Databricks, a regulator requires you to hold the files at a specific path you control. If none of those apply, choose managed.

## Converting between managed and external

You can flip a managed table to external (so UC stops owning the files), or convert an external table to managed (so UC takes ownership). Use the dedicated `ALTER TABLE ... SET MANAGED` / `SET EXTERNAL` syntax — covered in Section 7 of the exam.

```sql
 -- Take a managed table external (UC keeps metadata, but stops managing files)
 ALTER TABLE fintech_dev.silver.customers SET EXTERNAL;

 -- Adopt an external table as managed (UC moves files into its managed location)
 ALTER TABLE fintech_dev.silver.customers SET MANAGED;
```

The exam wants you to know:

- Both operations are **metadata-level on the UC side** — the table identity and three-part name stay the same.
- The *physical files* may or may not move depending on direction. Always test on a copy first.
- The principal performing the conversion needs the right privileges on both the table and the target storage location.

## Volumes — governed storage for files, not rows

Tables are for rows. **Volumes** are UC's answer for everything that isn't tabular — raw JSON dumps from the Payments source, PDFs of loan agreements, model artifacts, libraries, image datasets.

A volume is referenced like any UC object: `catalog.schema.volume_name`. From a notebook, you read and write its files at `/Volumes/catalog/schema/volume_name/...`.

Two flavours, mirroring tables:

- **Managed volume** — UC owns the storage location. Use for unstructured data your team produces.
- **External volume** — UC catalogs files at a path you already own. Use for data dropped in by external systems (the bank's SFTP landing zone for daily card files).

```sql
 CREATE VOLUME fintech_dev.bronze.landing_zone
   COMMENT 'Daily card extracts dropped here by the source system';

 -- Then from Python in a notebook:
 -- dbutils.fs.ls("/Volumes/fintech_dev/bronze/landing_zone")
```

Volumes replace the legacy `dbfs:/mnt/...` mount points. Mounts had no UC permission model; volumes do. **The exam expects volumes as the answer** whenever the question involves accessing files that aren't already Delta tables.

## The `hive_metastore` legacy catalog

Every workspace that pre-dates Unity Catalog still has a special catalog called `hive_metastore`. It is the *workspace-local* metastore — tables created with the legacy two-part name `schema.table` live there.

Two facts to lock in:

- `hive_metastore` is **per-workspace**, not regional. Two workspaces have two independent `hive_metastore` catalogs — there is no cross-workspace governance.
- New tables should **never** land in `hive_metastore`. Treat it as legacy. Use the **UCX** tool or `SYNC TABLE` to migrate its contents into a real UC catalog.

```sql
 -- Migrate a hive_metastore table to a UC catalog
 SYNC TABLE fintech_dev.silver.customers
   FROM hive_metastore.default.customers_old;
```

When the exam contrasts `hive_metastore.default.foo` vs. `fintech_dev.silver.foo`, the second one is always the right pattern.

## Hands-on — Delta + UC against `silver.card_transactions`

The cells below assume the `fintech_dev` catalog and `silver` schema you created at the end of notebook 01 already exist. We'll create the `card_transactions` table, walk through ACID + time travel + MERGE + maintenance, then peek at the underlying files.

Run this in a **Databricks notebook attached to a cluster** (DBR 14.x+) or a SQL warehouse. The `spark` session is provided by the workspace.

In [ ]:
# Set the working catalog/schema so we don't repeat fintech_dev.silver everywhere.
spark.sql("USE CATALOG fintech_dev")
spark.sql("USE SCHEMA silver")

In [ ]:
# Drop-and-create so the cell is rerunnable. Liquid Clustering on customer_id + transaction_at
# is the recommended modern layout for a per-customer time-series fact table.
spark.sql("DROP TABLE IF EXISTS card_transactions")
spark.sql("""
    CREATE TABLE card_transactions (
        transaction_id    STRING       NOT NULL,
        card_id           STRING       NOT NULL,
        customer_id       STRING       NOT NULL,
        merchant_name     STRING,
        merchant_category STRING,
        amount            DECIMAL(18, 2) NOT NULL,
        currency          STRING,
        transaction_at    TIMESTAMP    NOT NULL,
        status            STRING,
        is_flagged        BOOLEAN
    )
    USING DELTA
    CLUSTER BY (customer_id, transaction_at)
    COMMENT 'Conformed card transactions — joins customers via customer_id'
""")

In [ ]:
# First batch — three approved transactions. This is version 1 of the table.
spark.sql("""
    INSERT INTO card_transactions VALUES
      ('T001', 'CA001', 'C001', 'BigBasket',     'grocery', 1500.00, 'INR', TIMESTAMP '2024-01-05 10:30:00', 'approved', false),
      ('T002', 'CA002', 'C002', 'MakeMyTrip',    'travel',  8200.00, 'INR', TIMESTAMP '2024-01-06 14:10:00', 'approved', false),
      ('T003', 'CA003', 'C003', 'Shell Petrol',  'fuel',    3000.00, 'INR', TIMESTAMP '2024-01-07 09:00:00', 'approved', false)
""")

spark.sql("SELECT COUNT(*) AS row_count FROM card_transactions").show()

In [ ]:
# Late-arriving batch — a fraud-flagged transaction and a declined one. This is version 2.
spark.sql("""
    INSERT INTO card_transactions VALUES
      ('T004', 'CA001', 'C001', 'Unknown Merch', 'other',  45000.00, 'INR', TIMESTAMP '2024-01-05 23:55:00', 'approved', true),
      ('T005', 'CA002', 'C002', 'Amazon',        'retail',   500.00, 'INR', TIMESTAMP '2024-01-08 11:00:00', 'declined', false)
""")

In [ ]:
# DESCRIBE HISTORY shows every commit on the table — version, operation, user, timestamp.
# Each row corresponds to one NNNN.json file in _delta_log.
spark.sql("DESCRIBE HISTORY card_transactions").show(truncate=False)

In [ ]:
# Time travel — read the table as it was at version 1, BEFORE the fraud-flagged row landed.
before = spark.sql("SELECT COUNT(*) AS row_count_v1 FROM card_transactions VERSION AS OF 1")
before.show()

now = spark.sql("SELECT COUNT(*) AS row_count_current FROM card_transactions")
now.show()

In [ ]:
# Schema enforcement — try to insert a row with an extra column. Delta refuses.
from pyspark.sql import Row

rogue = spark.createDataFrame([
    Row(transaction_id="T999", card_id="CA999", customer_id="C999",
        merchant_name="Test", merchant_category="misc",
        amount=100.00, currency="INR",
        transaction_at="2024-01-09 10:00:00", status="approved", is_flagged=False,
        merchant_country="IN")     # <-- the extra column
])

try:
    rogue.write.mode("append").saveAsTable("card_transactions")
except Exception as e:
    print("Rejected — schema enforcement worked:")
    print(str(e).splitlines()[0])

In [ ]:
# Schema evolution — same write, this time with mergeSchema = true. Delta adds the column.
rogue.write.mode("append").option("mergeSchema", "true").saveAsTable("card_transactions")

spark.sql("DESCRIBE TABLE card_transactions").show(truncate=False)

In [ ]:
# MERGE INTO — late-arriving correction for T002 (the merchant should be 'Cleartrip', not 'MakeMyTrip')
# plus a brand-new transaction T006. One statement handles both update and insert.
spark.sql("""
    CREATE OR REPLACE TEMP VIEW corrections AS
    SELECT * FROM VALUES
      ('T002', 'CA002', 'C002', 'Cleartrip',  'travel', 8200.00, 'INR', TIMESTAMP '2024-01-06 14:10:00', 'approved', false),
      ('T006', 'CA003', 'C003', 'Flipkart',   'retail', 1200.00, 'INR', TIMESTAMP '2024-01-09 12:00:00', 'approved', false)
    AS v(transaction_id, card_id, customer_id, merchant_name, merchant_category,
         amount, currency, transaction_at, status, is_flagged)
""")

spark.sql("""
    MERGE INTO card_transactions AS t
    USING corrections           AS s
       ON t.transaction_id = s.transaction_id
    WHEN MATCHED THEN UPDATE SET
        merchant_name = s.merchant_name
    WHEN NOT MATCHED THEN INSERT (
        transaction_id, card_id, customer_id, merchant_name, merchant_category,
        amount, currency, transaction_at, status, is_flagged
    ) VALUES (
        s.transaction_id, s.card_id, s.customer_id, s.merchant_name, s.merchant_category,
        s.amount, s.currency, s.transaction_at, s.status, s.is_flagged
    )
""")

spark.sql("""
    SELECT transaction_id, merchant_name, amount
    FROM card_transactions
    WHERE transaction_id IN ('T002', 'T006')
    ORDER BY transaction_id
""").show(truncate=False)

In [ ]:
# DELETE — drop the declined row T005. On supported runtimes this uses a deletion vector,
# so the underlying Parquet file is NOT immediately rewritten.
spark.sql("DELETE FROM card_transactions WHERE transaction_id = 'T005'")

spark.sql("""
    SELECT COUNT(*) AS remaining_rows
    FROM card_transactions
""").show()

In [ ]:
# OPTIMIZE compacts small files into larger ones. On a Liquid-Clustered table, it also
# re-balances clusters based on the declared keys.
spark.sql("OPTIMIZE card_transactions").show(truncate=False)

In [ ]:
# DESCRIBE DETAIL exposes the physical truth — storage location (UC-managed), the format,
# file count, total size, and the clustering columns.
spark.sql("DESCRIBE DETAIL card_transactions").show(truncate=False, vertical=True)

In [ ]:
# A volume for raw landing files — the SFTP drop zone for the Cards source extract.
# Note: dbutils.fs is the Databricks file utility; on classic compute it works against /Volumes/...
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("""
    CREATE VOLUME IF NOT EXISTS fintech_dev.bronze.landing_zone
    COMMENT 'Daily card extracts dropped here by the source system'
""")

spark.sql("SHOW VOLUMES IN fintech_dev.bronze").show(truncate=False)

## Section 1 + 7 self-check

Five exam-style questions on this notebook's material. Answers at the bottom.

**1.** Which statement most accurately describes how a Delta table provides ACID transactions on object storage?

- A. The underlying object store provides multi-object transactions
- B. Delta locks the directory during writes
- C. Each transaction commits by atomically appending a new JSON commit file to `_delta_log`; readers compose the log to see a consistent snapshot
- D. Delta enforces transactions by writing a single Parquet file per commit

**2.** A data engineer needs to roll back the `silver.customers` table to its state from 24 hours ago after a bad backfill. Which operation does this in a single command?

- A. `RESTORE TABLE ... TO TIMESTAMP AS OF ...`
- B. `DROP TABLE` and reload from source
- C. `VACUUM`
- D. `OPTIMIZE`

**3.** Which feature does a UC **managed** table get that a UC **external** table does not?

- A. Time travel
- B. Schema enforcement
- C. Predictive Optimization
- D. ACID transactions

**4.** Which file format / metadata layout does Unity Catalog use to make a Delta table also readable by Iceberg-native engines, without duplicating data?

- A. Delta UniForm
- B. Photon
- C. Liquid Clustering
- D. `hive_metastore`

**5.** The bank stores daily PDF loan agreements arriving from a third-party. Which UC object type is correct for governing access to these files?

- A. A managed Delta table
- B. A view
- C. A volume
- D. A function

<details><summary>Show answers</summary>

1. **C** — atomic JSON commit append + log replay is the mechanism.
2. **A** — `RESTORE TABLE ... TO TIMESTAMP AS OF` rewinds the table state in one command.
3. **C** — Predictive Optimization runs `OPTIMIZE` and `VACUUM` automatically, and is managed-only.
4. **A** — Delta UniForm writes Iceberg metadata alongside Delta, pointing at the same Parquet files.
5. **C** — Volumes are UC's governed object for files that aren't rows.

</details>